# NLP-to-SQL / SQL-to-MQL: Fine-tune T5-large (continued from best checkpoint)
**Runtime → Change runtime type → A100 GPU** before running.

Warm-start from the saved best model (`t5-large-sql-mql/best`). Two tasks:
- **Spider** (NL → SQL, ~7 000 train examples), prefix `translate to sql:`
- **DocSpider** (SQL → MQL, pairs built by joining Spider + DocSpider on question text), prefix `translate sql to mql:`

5 epochs at lr=1e-5 (lower than the original 3e-5 — model is already warm).

In [ ]:
# 1. Check GPU
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM: {total:.1f} GB')

In [ ]:
# 2. Install missing packages
!pip install -q sentencepiece accelerate

In [ ]:
# 3. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 4. Unzip Spider data to local Colab storage
import os
SPIDER_DIR = '/content/spider_data/spider_data'

if not os.path.exists(SPIDER_DIR):
    print('Unzipping Spider data...')
    !unzip -q /content/drive/MyDrive/AIML_26/CapstoneProject/spider_data.zip -d /content/
    print('Done.')
else:
    print('Spider data already extracted.')

for f in ['tables.json', 'train_spider.json', 'dev.json']:
    path = f'{SPIDER_DIR}/{f}'
    print(f'{f}: {"OK" if os.path.exists(path) else "MISSING"}')

In [ ]:
# 4b. Extract DocSpider data to local Colab storage
DOCSPIDER_DIR = '/content/docspider_data'

if not os.path.exists(DOCSPIDER_DIR):
    print('Extracting DocSpider data...')
    !unzip -q /content/drive/MyDrive/AIML_26/CapstoneProject/docspider_data.zip -d /content/
    print('Done.')
else:
    print('DocSpider data already extracted.')

for f in ['collections.json', 'train.json', 'dev.json']:
    path = f'{DOCSPIDER_DIR}/{f}'
    print(f'{f}: {"OK" if os.path.exists(path) else "MISSING"}')

In [ ]:
# 5. Schema serialization for both Spider (SQL) and DocSpider (MQL)
import json

def build_sql_schema_lookup(tables_path):
    with open(tables_path) as f:
        tables = json.load(f)
    lookup = {}
    for db in tables:
        db_id     = db['db_id']
        tbl_names = db['table_names_original']
        col_names = db['column_names_original']
        tbl_cols  = {i: [] for i in range(len(tbl_names))}
        for tbl_idx, col_name in col_names:
            if tbl_idx == -1:
                continue
            tbl_cols[tbl_idx].append(col_name)
        parts = [db_id]
        for i, tname in enumerate(tbl_names):
            parts.append(f'{tname} : {" , ".join(tbl_cols[i])}')
        lookup[db_id] = ' | '.join(parts)
    return lookup

def build_mql_schema_lookup(collections_path):
    with open(collections_path) as f:
        collections = json.load(f)
    lookup = {}
    for db in collections:
        db_id      = db['db_id']
        coll_names = db['collection_names']
        col_names  = db['column_names']
        coll_cols  = {i: [] for i in range(len(coll_names))}
        for coll_idx, col_name in col_names:
            if coll_idx == -1:
                continue
            coll_cols[coll_idx].append(col_name)
        parts = [db_id]
        for i, cname in enumerate(coll_names):
            parts.append(f'{cname} : {" , ".join(coll_cols[i])}')
        lookup[db_id] = ' | '.join(parts)
    return lookup

sql_schema_lookup = build_sql_schema_lookup(f'{SPIDER_DIR}/tables.json')
mql_schema_lookup = build_mql_schema_lookup(f'{DOCSPIDER_DIR}/collections.json')

print(f'SQL schema lookup : {len(sql_schema_lookup)} databases')
print(f'MQL schema lookup : {len(mql_schema_lookup)} databases')

In [ ]:
# 6. Dataset classes
#
# NL2SQLDataset  : NL question → SQL query  (Spider, prefix 'translate to sql:')
# SQL2MQLDataset : SQL query   → MQL query  (DocSpider only — spider_gold_sql field
#                                             already contains the paired SQL query,
#                                             no join with Spider needed)

import json
from torch.utils.data import Dataset

class NL2SQLDataset(Dataset):
    def __init__(self, data_path, schema_lookup, tokenizer,
                 max_input_len, max_target_len):
        with open(data_path) as f:
            self.data = json.load(f)
        self.schema_lookup  = schema_lookup
        self.tokenizer      = tokenizer
        self.max_input_len  = max_input_len
        self.max_target_len = max_target_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        example    = self.data[idx]
        schema     = self.schema_lookup.get(example['db_id'], example['db_id'])
        input_text = f"translate to sql: {example['question']} | {schema}"
        model_inputs = self.tokenizer(
            input_text, max_length=self.max_input_len, truncation=True)
        labels = self.tokenizer(
            text_target=example['query'], max_length=self.max_target_len, truncation=True)
        model_inputs['labels'] = labels['input_ids']
        return model_inputs


class SQL2MQLDataset(Dataset):
    def __init__(self, data_path, mql_schema_lookup,
                 tokenizer, max_input_len, max_target_len):
        with open(data_path) as f:
            self.data = json.load(f)
        self.mql_schema_lookup = mql_schema_lookup
        self.tokenizer         = tokenizer
        self.max_input_len     = max_input_len
        self.max_target_len    = max_target_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        example    = self.data[idx]
        schema     = self.mql_schema_lookup.get(example['db_id'], example['db_id'])
        input_text = f"translate sql to mql: {example['spider_gold_sql']} | {schema}"
        model_inputs = self.tokenizer(
            input_text, max_length=self.max_input_len, truncation=True)
        labels = self.tokenizer(
            text_target=example['query'], max_length=self.max_target_len, truncation=True)
        model_inputs['labels'] = labels['input_ids']
        return model_inputs

In [ ]:
# 7. Evaluation metric — Exact Match Accuracy (works for both SQL and MQL)
import re
import numpy as np

def normalize_query(query):
    query = query.lower().strip()
    query = re.sub(r'\s+', ' ', query)
    query = re.sub(r'\s*,\s*', ', ', query)
    query = re.sub(r'\s*(=|!=|>=|<=|>|<)\s*', r' \1 ', query)
    return query

def compute_metrics(tokenizer):
    def _compute(eval_pred):
        predictions, labels = eval_pred
        predictions = np.where(predictions != -100, predictions, tokenizer.pad_token_id)
        decoded_preds  = tokenizer.batch_decode(predictions, skip_special_tokens=True)
        labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
        decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
        exact_matches = [
            normalize_query(p) == normalize_query(l)
            for p, l in zip(decoded_preds, decoded_labels)
        ]
        return {'exact_match': round(np.mean(exact_matches), 4)}
    return _compute

In [ ]:
# 8. Load model and tokenizer from saved best checkpoint
# Warm-start: weights from the best NL→SQL / NL→MQL model.
# Optimizer and scheduler are NOT loaded — fresh training state.
from transformers import AutoTokenizer, T5ForConditionalGeneration

BEST_MODEL_PATH = '/content/drive/MyDrive/AIML_26/CapstoneProject/t5-large-sql-mql/best'

tokenizer = AutoTokenizer.from_pretrained(BEST_MODEL_PATH)
model     = T5ForConditionalGeneration.from_pretrained(BEST_MODEL_PATH)

total_params = sum(p.numel() for p in model.parameters())
print(f'Model loaded from: {BEST_MODEL_PATH}')
print(f'Total parameters : {total_params/1e6:.0f}M')

In [ ]:
# 9. Freeze all layers, then unfreeze last N encoder + decoder blocks and LM head
N_UNFREEZE = 4
NUM_LAYERS  = 24

for param in model.parameters():
    param.requires_grad = False

for i in range(NUM_LAYERS - N_UNFREEZE, NUM_LAYERS):
    for param in model.encoder.block[i].parameters():
        param.requires_grad = True

for i in range(NUM_LAYERS - N_UNFREEZE, NUM_LAYERS):
    for param in model.decoder.block[i].parameters():
        param.requires_grad = True

for param in model.encoder.final_layer_norm.parameters():
    param.requires_grad = True
for param in model.decoder.final_layer_norm.parameters():
    param.requires_grad = True
for param in model.lm_head.parameters():
    param.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable: {trainable/1e6:.1f}M / {total/1e6:.0f}M ({100*trainable/total:.1f}%)')

In [ ]:
# 10. Build datasets
# Task 1: NL → SQL  (Spider)
# Task 2: SQL → MQL (DocSpider — spider_gold_sql → query)
from torch.utils.data import ConcatDataset

MAX_INPUT_LEN  = 512
MAX_TARGET_LEN = 256

sql_train = NL2SQLDataset(
    f'{SPIDER_DIR}/train_spider.json', sql_schema_lookup,
    tokenizer, MAX_INPUT_LEN, MAX_TARGET_LEN)

sql_eval = NL2SQLDataset(
    f'{SPIDER_DIR}/dev.json', sql_schema_lookup,
    tokenizer, MAX_INPUT_LEN, MAX_TARGET_LEN)

mql_train = SQL2MQLDataset(
    f'{DOCSPIDER_DIR}/train.json', mql_schema_lookup,
    tokenizer, MAX_INPUT_LEN, MAX_TARGET_LEN)

mql_eval = SQL2MQLDataset(
    f'{DOCSPIDER_DIR}/dev.json', mql_schema_lookup,
    tokenizer, MAX_INPUT_LEN, MAX_TARGET_LEN)

train_dataset = ConcatDataset([sql_train, mql_train])
eval_dataset  = ConcatDataset([sql_eval,  mql_eval])

print(f'SQL  train: {len(sql_train):5d}  eval: {len(sql_eval)}')
print(f'MQL  train: {len(mql_train):5d}  eval: {len(mql_eval)}')
print(f'Combined   train: {len(train_dataset)}  eval: {len(eval_dataset)}')

# Show a sample SQL→MQL pair to verify the data loaded correctly
sample = mql_train[0]
with open(f'{DOCSPIDER_DIR}/train.json') as f:
    _d = json.load(f)[0]
print(f'\nSample SQL→MQL pair:')
print(f'  SQL : {_d["spider_gold_sql"]}')
print(f'  MQL : {_d["query"]}')

In [ ]:
# 11. Training configuration
# save_total_limit=1  — only the single most recent checkpoint is kept on Drive at any time.
# load_best_model_at_end=True — after all epochs, the best checkpoint is reloaded into memory
#   before Cell 13 saves it. The checkpoint directory is deleted in Cell 13 after saving.
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq

OUTPUT_DIR = '/content/drive/MyDrive/AIML_26/CapstoneProject/t5-large-sql-mql-final'

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer, model=model,
    label_pad_token_id=-100, pad_to_multiple_of=8)

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=8,
    learning_rate=1e-5,
    warmup_steps=50,
    weight_decay=0.01,
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LEN,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='exact_match',
    greater_is_better=True,
    save_total_limit=1,
    bf16=True,
    logging_steps=50,
    report_to='none',
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics(tokenizer),
)
print('Trainer ready.')

In [ ]:
# 11b. Pre-training diagnostic
import torch

nan_params = [(n, p) for n, p in model.named_parameters() if torch.isnan(p).any()]
print(f'[1] NaN weight tensors: {len(nan_params)}')
if nan_params:
    print('    !! STOP: re-run Cell 8 to reload clean weights.')
else:
    print('    OK')

trainable_params = [(n, p) for n, p in model.named_parameters() if p.requires_grad]
trainable_count  = sum(p.numel() for _, p in trainable_params)
print(f'[2] Trainable tensors: {len(trainable_params)}  |  {trainable_count/1e6:.1f}M params')
if not trainable_params:
    print('    !! STOP: re-run Cell 9 to set requires_grad.')

model.eval()
with torch.no_grad():
    sample    = train_dataset[0]
    input_ids = torch.tensor([sample['input_ids']]).to(model.device)
    attn_mask = torch.tensor([sample['attention_mask']]).to(model.device)
    labels    = torch.tensor([sample['labels']]).to(model.device)
    out = model(input_ids=input_ids, attention_mask=attn_mask, labels=labels)
    loss_val = out.loss.item()

print(f'[3] Single-sample loss: {loss_val:.4f}')
if torch.isnan(out.loss):
    print('    !! STOP: NaN loss — re-run Cell 8.')
elif loss_val < 0.01:
    print('    !! WARNING: loss near 0 — labels may all be masked.')
else:
    print('    OK — proceed to training.')
model.train()

In [ ]:
# 12. Train
trainer.train()

In [ ]:
# 13. Save best model to Drive, then delete checkpoint directory
import shutil, glob

BEST_MODEL_DIR = f'{OUTPUT_DIR}/best'
trainer.save_model(BEST_MODEL_DIR)
tokenizer.save_pretrained(BEST_MODEL_DIR)
print(f'Best model saved to {BEST_MODEL_DIR}')

# Delete all checkpoint-* directories — best model is now in best/ and Drive space is freed
checkpoint_dirs = glob.glob(f'{OUTPUT_DIR}/checkpoint-*')
for ckpt_dir in checkpoint_dirs:
    shutil.rmtree(ckpt_dir)
    print(f'Deleted: {ckpt_dir}')
print('Done — only best/ remains on Drive.')

In [ ]:
# 14. Inference test
# SQL task : NL question → SQL query   (same as before)
# MQL task : SQL query   → MQL query   (uses spider_gold_sql from DocSpider dev)
import json
from transformers import T5ForConditionalGeneration, AutoTokenizer

BEST_MODEL_DIR = '/content/drive/MyDrive/AIML_26/CapstoneProject/t5-large-sql-mql-final/best'
MAX_INPUT_LEN  = 512
MAX_TARGET_LEN = 256

# Load model and tokenizer from best/ if not already in memory
try:
    model
    tokenizer
    print('Using model already in memory.')
except NameError:
    print(f'Model not in memory — loading from {BEST_MODEL_DIR}')
    tokenizer = AutoTokenizer.from_pretrained(BEST_MODEL_DIR)
    model     = T5ForConditionalGeneration.from_pretrained(BEST_MODEL_DIR)
    model.eval()
    print('Model loaded.')

import re
def normalize_query(query):
    query = query.lower().strip()
    query = re.sub(r'\s+', ' ', query)
    query = re.sub(r'\s*,\s*', ', ', query)
    query = re.sub(r'\s*(=|!=|>=|<=|>|<)\s*', r' \1 ', query)
    return query

def generate_query(input_text):
    inputs = tokenizer(input_text, return_tensors='pt',
                       max_length=MAX_INPUT_LEN, truncation=True)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    outputs = model.generate(**inputs, max_new_tokens=MAX_TARGET_LEN, num_beams=4)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# --- SQL (Spider dev — NL → SQL) ---
print('=' * 60)
print('SQL  (Spider dev — NL → SQL, first 5 examples)')
print('=' * 60)
with open(f'{SPIDER_DIR}/dev.json') as f:
    sql_dev = json.load(f)

sql_correct = 0
for ex in sql_dev[:5]:
    schema     = sql_schema_lookup[ex['db_id']]
    input_text = f"translate to sql: {ex['question']} | {schema}"
    predicted  = generate_query(input_text)
    gold       = ex['query']
    match      = normalize_query(predicted) == normalize_query(gold)
    sql_correct += match
    print(f'Q         : {ex["question"]}')
    print(f'Gold      : {gold}')
    print(f'Predicted : {predicted}')
    print(f'Match: {"YES" if match else "NO"}\n')
print(f'SQL exact match: {sql_correct}/5')

# --- MQL (DocSpider dev — SQL → MQL) ---
print()
print('=' * 60)
print('MQL  (DocSpider dev — SQL → MQL, first 5 examples)')
print('=' * 60)
with open(f'{DOCSPIDER_DIR}/dev.json') as f:
    mql_dev = json.load(f)

mql_correct = 0
for ex in mql_dev[:5]:
    schema     = mql_schema_lookup.get(ex['db_id'], ex['db_id'])
    input_text = f"translate sql to mql: {ex['spider_gold_sql']} | {schema}"
    predicted  = generate_query(input_text)
    gold       = ex['query']
    match      = normalize_query(predicted) == normalize_query(gold)
    mql_correct += match
    print(f'SQL       : {ex["spider_gold_sql"]}')
    print(f'Gold MQL  : {gold}')
    print(f'Predicted : {predicted}')
    print(f'Match: {"YES" if match else "NO"}\n')
print(f'MQL exact match: {mql_correct}/5')

In [ ]:
#Recoveory cell
import shutil, glob
from transformers import T5ForConditionalGeneration, AutoTokenizer

OUTPUT_DIR    = '/content/drive/MyDrive/AIML_26/CapstoneProject/t5-large-sql-mql-final'
BEST_MODEL_DIR = f'{OUTPUT_DIR}/best'

# Find the surviving checkpoint
checkpoint_dirs = glob.glob(f'{OUTPUT_DIR}/checkpoint-*')
print('Found checkpoints:', checkpoint_dirs)

# Load and save as best model
ckpt_path = checkpoint_dirs[0]
model     = T5ForConditionalGeneration.from_pretrained(ckpt_path)
tokenizer = AutoTokenizer.from_pretrained(ckpt_path)
model.save_pretrained(BEST_MODEL_DIR)
tokenizer.save_pretrained(BEST_MODEL_DIR)
print(f'Saved to {BEST_MODEL_DIR}')

# Delete checkpoint
shutil.rmtree(ckpt_path)
print(f'Deleted {ckpt_path}')